<a href="https://colab.research.google.com/github/21f1000895-png/Langchain_Rag_Workshop_iitm/blob/main/Assign_of_workshop_langchain_rag2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langchain langchain-core langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.4/80.4 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
!pip install -q langchain-groq langchain-huggingface sentence-transformers faiss-cpu pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 405.1/405.1 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.5 MB/s eta 0:00:00


In [3]:
from google.colab import userdata
from langchain_groq import ChatGroq

llm = ChatGroq (
    model = "openai/gpt-oss-120b",
    api_key = userdata.get("Groq_api_key")
)

In [4]:
response = llm.invoke("what is langchain? answer in one line only")
response.content

'LangChain is a Python framework that simplifies building applications powered by language models by chaining together prompts, models, and data sources.'

In [5]:
response = llm.invoke("what new product launched by acme inc in 2026?")
response.content

'I’m sorry, but I don’t have information about products that Acme\u202fInc. may have launched in 2026. My training only includes knowledge up to June\u202f2024, and I don’t have the ability to browse the web for newer data.'

#Rag
steps:
1. load the data for retrieval.
2. Split the data or documents.
3. produce the embeddings for each of the splits.
4. store the embeddings in a vecator db
5. create a retriever to retrive the embeddings from the vector db
6. combine the llm and the retriever and produce the results.

In [6]:
#load text file
from langchain_community.document_loaders import TextLoader
loader = TextLoader("technova_policy.txt")
doc1 = loader.load()
doc1

/tmp/ipykernel_2021/2074186784.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


[Document(metadata={'source': 'technova_policy.txt'}, page_content="\ufeffTechNova Solutions - Employee Policy Document\n\n\n1. Annual Leave: Every full-time employee receives 18 paid annual leave days per calendar year. Employees must submit annual leave requests to their manager at least 3 calendar days before the requested leave date. Leave approval depends on the manager's confirmation.\n\n\n2. Work From Home: Employees may work remotely on Wednesdays and Fridays. Employees must inform their manager by email at least one working day in advance. Remote work is subject to manager approval.\n\n\n3. Working Hours: The standard working day is from 9:30 AM to 5:30 PM, Monday through Friday. Employees receive a one-hour lunch break.\n\n\n4. Sick Leave: Employees receive 10 paid sick leave days per calendar year. If an employee is absent due to illness for more than two consecutive working days, a medical certificate must be submitted to HR.\n\n\n5. Learning Allowance: Each full-time emplo

#Split


In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
#recursive character text splitter splits word by word repecting word boundary.
splitter = RecursiveCharacterTextSplitter(chunk_size = 50, chunk_overlap= 15)
chunks = splitter.split_documents(doc1)
chunks

[Document(metadata={'source': 'technova_policy.txt'}, page_content='\ufeffTechNova Solutions - Employee Policy Document'),
 Document(metadata={'source': 'technova_policy.txt'}, page_content='1. Annual Leave: Every full-time employee'),
 Document(metadata={'source': 'technova_policy.txt'}, page_content='employee receives 18 paid annual leave days per'),
 Document(metadata={'source': 'technova_policy.txt'}, page_content='leave days per calendar year. Employees must'),
 Document(metadata={'source': 'technova_policy.txt'}, page_content='Employees must submit annual leave requests to'),
 Document(metadata={'source': 'technova_policy.txt'}, page_content='requests to their manager at least 3 calendar'),
 Document(metadata={'source': 'technova_policy.txt'}, page_content='3 calendar days before the requested leave date.'),
 Document(metadata={'source': 'technova_policy.txt'}, page_content='leave date. Leave approval depends on the'),
 Document(metadata={'source': 'technova_policy.txt'}, page_co

#Embeddings
converting the chunks into embeddins. sentence tranformers are used to create embeddings


In [8]:
from langchain_huggingface import HuggingFaceEmbeddings
embedding_model = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

#store embeddings in vectorDB

In [9]:
from langchain_community.vectorstores import FAISS
vectorstore = FAISS.from_documents(chunks, embedding_model)


In [10]:
retrieved_docs = vectorstore.similarity_search("how many sick leave?", k = 2)
retrieved_docs

[Document(id='82e02202-8d97-405f-b1f7-70f9d4aa6d64', metadata={'source': 'technova_policy.txt'}, page_content='4. Sick Leave: Employees receive 10 paid sick'),
 Document(id='c144366b-e9b3-4e7e-9869-a2fe34c32153', metadata={'source': 'technova_policy.txt'}, page_content='10 paid sick leave days per calendar year. If an')]

# Create Retriever

In [11]:
retriever = vectorstore.as_retriever(search_kwargs = {"k": 2})

# Create RAG chain

In [13]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [14]:
def format_docs(docs):
  return "\n\n".join(doc.page_content for doc in docs)

In [15]:
template = """
Use the following context to answer the question.
Use three sentences maximum and keep the answer concise.

Context: {context}

Question: {question}

Answer:
"""

prompt = PromptTemplate.from_template(template)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [16]:
query = "How many paid annual leave days does a full-time employee receive per year?"

response = rag_chain.invoke(query)
response

'A full‑time employee receives **18 paid annual leave days per year**.'

In [17]:
query = "An employee wants to work remotely on Wednesday and take leave on Friday. What do the company policies say about these two requests?"

response = rag_chain.invoke(query)
response

'The policy permits employees to work remotely on Wednesdays, so that request is allowed. Fridays are also designated as remote‑work days, so taking a regular leave instead would require a separate leave approval. In short, the Wednesday remote request is fine; the Friday leave must be processed through the company’s leave‑of‑absence procedure.'

In [18]:
query = "What is the company's parental leave policy?"

response = rag_chain.invoke(query)
response

'I’m sorry, but the provided context does not include any information about the company’s parental leave policy.'